In [0]:
%sql
MERGE INTO catalog.gold.fact_ventas AS tgt
USING (

    SELECT
        v.venta,
        f.id_fecha,
        p.id_articulo,
        c.id_cliente,
        u.id_usulogin,
        t.id_turno,
        m.id_condvtapos,
        dd.id_delivery,
        v.vtaestado,
        v.precio,
        v.cant,
        v.total,
        CURRENT_TIMESTAMP() AS _refresh_timestamp

    FROM catalog_ventas.silver.ventas_clean v

    
    LEFT JOIN catalog_ventas.gold.dim_calendario f
        ON f.id_fecha = CAST(DATE_FORMAT(v.vtafecha,'yyyyMMdd') AS BIGINT)

    LEFT JOIN catalog_ventas.gold.dim_producto p
        ON p.articulo = v.articulo

    LEFT JOIN catalog_ventas.gold.dim_cliente c
        ON c.cliente = v.cliente

    LEFT JOIN catalog_ventas.gold.dim_cajero u
        ON u.usulogin = v.usulogin

    LEFT JOIN catalog_ventas.gold.dim_turno t
        ON t.turno = v.turno

    LEFT JOIN catalog_ventas.gold.dim_condvtapos m
        ON m.condvtapos = v.condvtapos
    
    LEFT JOIN catalog.gold.dim_delivery dd 
        ON v.delivery = dd.delivery

    WHERE v.vtafecha IS NOT NULL

) AS src


ON tgt.venta = src.venta
AND tgt.id_articulo = src.id_articulo

--  update solo si cambia el estado (opcional)
WHEN MATCHED AND tgt.vtaestado <> src.vtaestado THEN
UPDATE SET
    tgt.vtaestado = src.vtaestado,
    tgt._refresh_timestamp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
INSERT (
    venta,
    id_fecha,
    id_articulo,
    id_cliente,
    id_usulogin,
    id_turno,
    id_condvtapos,
    id_delivery,
    vtaestado,
    precio,
    cant,
    total,
    _refresh_timestamp
)
VALUES (
    src.venta,
    src.id_fecha,
    src.id_articulo,
    src.id_cliente,
    src.id_usulogin,
    src.id_turno,
    src.id_condvtapos,
    src.id_delivery,
    src.vtaestado,
    src.precio,
    src.cant,
    src.total,
    src._refresh_timestamp
);